<a href="https://colab.research.google.com/github/Precious00001/GenerativeAI-101/blob/main/medical-research-assistant/v1.0/Medical_Research_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-google-genai google-generativeai tiktoken langchain-community
!pip install langchain langchain-google-genai google-generativeai tiktoken faiss-cpu huggingface_hub langchain-community langgraph
!pip install -U langchain-groq
!pip install -U langchain-huggingface sentence-transformers
!pip install -U langchain-community faiss-cpu
!pip install tavily-python
!pip install pypdf
!pip install langchain-text-splitters
!pip install duckduckgo-search
!pip install -U ddgs

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent
import uuid
import time
from google.colab import userdata

In [ ]:
#LLM
groq_api_key = userdata.get("groq_api_key")
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=groq_api_key,
    temperature=0,
    max_tokens=500,
    max_retries=3,  # retry automatically
    request_timeout=60
)

In [ ]:
#Searching tool
search_tool = DuckDuckGoSearchResults()

#Vectorstore (will set in main)
vectorstore = None

In [ ]:
def load_pdfs(pdf_paths):
  all_text = ""
  for path in pdf_paths:
    loader = PyPDFLoader(path)
    documents = loader.load()
    for doc in documents:
        all_text += doc.page_content + "\n"
    print(f"Loaded: {path}")
  return all_text


def chunk_text(text, chunk_size=500):
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunk = text[i:i+chunk_size]
    if len(chunk) > 200:
      chunks.append(chunk)
  return chunks or [text]

# Load embeddings once at the top level, not inside the function
print("Loading embeddings model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embeddings model loaded!")

def create_vector_store(texts):
  # Convert raw text chunks into LangChain Document objects
    documents = [
        Document(
            page_content=text,
            metadata={"id": str(uuid.uuid4())}
        ) for text in texts
    ]

    # Use the already loaded embeddings instead of reloading
    vector_store = FAISS.from_documents(documents, embeddings)

    return vector_store

In [ ]:
@tool
def search_documents(query: str) -> str:
  """Search for relevant information from the loaded PDF documents."""

  #Check if vector store has been created
  if vectorstore is None:
    return "Vector store not available."

  #Search for similar chunks in vector store
  docs_with_scores = vectorstore.similarity_search_with_score(query, k=3)

  # Filter out chunks that are not relevant enough
  # Lower score = more similar, so we keep score below 1.5
  relevant_docs = [doc for doc, score in docs_with_scores if score <1.5]

  if not relevant_docs:
    return "No relevant docs found."

  #combine relevant chunks into a single string
  combined = "\n".join([f"- {doc.page_content}" for doc in relevant_docs])
  return f"Source: PDF Document\n\nRelevant documents:\n{combined}"


@tool
def dosage_calculator(weight_and_dose: str) -> str:
  """ Calculate total dosage given patient weight and dose per kg.
  Input format: 'wieght_kg, dose_per_kg, e.g. '70, 5'
  """
  try:
    #split the input string into weight and dose
    parts = weight_and_dose.split(",")
    weight = float(parts[0].strip())
    dose_per_kg = float(parts[1].strip())

    #Calculate tottal dosa
    total_dose = weight * dose_per_kg

    return f"Source: Dosage Calculator\n\nTotal dosage: {total_dose}mg (Weight: {weight}kg × Dose: {dose_per_kg}mg/kg)"
  except Exception as e:
    return f"Error calculating dosage: {str(e)}"

@tool
def search_drug_interactions(drugs: str) -> str:
    """Search for interactions between specified drugs.
    Input format: 'drug1, drug2' e.g. 'aspirin, ibuprofen'
    """
    try:
        # DuckDuckGo returns a single string directly
        results = search_tool.invoke({"query": f"drug interactions between {drugs}"})
        return f"Source: Internet Search\n\n{results}"
    except Exception as e:
        return f"Error searching drug interactions: {str(e)}"

In [ ]:
system_prompt = """You are a medical research assistant. You ONLY have access to these EXACT tools:
1. search_documents
2. dosage_calculator
3. search_drug_interactions

STRICT RULES:
1. NEVER call any tool not listed above
2. NEVER use brave_search, google_search or any other search tool
3. Use ONE tool only once per question
4. After getting the tool result, give your final answer immediately
5. Do NOT keep reasoning after you have an answer
6. ALWAYS format your final answer EXACTLY like this:

Answer: [your answer here]
Source: [where the information came from]

Never skip the Source line. Never skip the Answer line."""

def main():
    global vectorstore

    # --- Load and index PDF documents FIRST ---
    try:
        pdf_paths = ["/content/Introduction-to-research-and-ethics-Iveta-6-Feb.pdf"]
        pdf_text = load_pdfs(pdf_paths)
        chunks = chunk_text(pdf_text)
        chunks = chunks[:20] # limit during dev to avoid rate limits
        vectorstore = create_vector_store(chunks)
        print("Vector store created successfully!")
    except Exception as e:
        print(f"Vector store creation failed: {str(e)}")
        vectorstore = None

    # --- Set up tools and agent AFTER vector store ---
    tools = [search_documents, dosage_calculator, search_drug_interactions]
    agent = create_react_agent(llm, tools, prompt=system_prompt)

    # --- Test questions ---
    questions = [
        "What should I do before I start research?",
        "What is the dosage for a 70kg patient at 5mg/kg?",
        "What are the interactions between aspirin and ibuprofen?"
    ]

    # --- Run each question through the agent ---
    for question in questions:
        print(f"\nQuestion: {question}")
        response = agent.invoke(
            {"messages": [("human", question)]},
            config={"recursion_limit": 15}
        )
        print(response['messages'][-1].content)
    print("-" * 50)
    time.sleep(2)

main()

In [ ]:
# Quick diagnostic - run this separately
results = search_tool.invoke({"query": "drug interactions between aspirin and ibuprofen"})
print(type(results))
print(results)